# Suite QRY — Queries

A rule is a query: `select(name, rule, variables)` streams a store's objects of a schema (its extent: what its
singletons reach) for which the rule holds. `Queries.Scan(store)` makes any store queryable in memory.

In [ ]:
from mbse.Expressions import Expressions as E
from mbse.Patterns import Queries as Q
from mbse.Schemas.Framework import Bindings as Bd, Schemas as S, Stores
from support import Contact, book, listed, native, protocol_problems, raises

this = E.variable("this")
store = book()
ann = store.Contact().name("ann").age(30).create()
bob = store.Contact().name("bob").create()
cy = store.Contact().name("cy").age(9).create()
listed(store, ann, bob, cy)
loose = store.Contact().name("dee").age(50).create()  # never listed: not the store's data
query = Q.Scan(store)

## QRY-01 · A query streams the extent's objects for which its rule holds

In [ ]:
adults = query.select("Contact", this.age.ge(18))
assert iter(adults) is adults and next(adults) is ann  # an iterator, read as it is asked
assert list(adults) == []  # the rest: bob's age is unknown, cy is a child, dee is not in the extent
assert list(query.select("Contact", this.age.ge(18), unknown=True)) == [ann, bob]
assert list(query.select("Contact", this.has("name"))) == [ann, bob, cy]

## QRY-02 · Variables are bound once, and objects too

In [ ]:
assert list(query.select("Contact", this.age.ge(E.variable("min")), {"min": 21})) == [ann]
assert list(query.select("Contact", this.age.lt(E.variable("max")).and_(this.ne(E.variable("not_"))),
            {"max": 40, "not_": cy})) == [ann]  # an object has no literal: it stays a variable, bound per object
assert list(query.select("Contact", E.variable("flag").or_(this.age.ge(99)), {"flag": True})) == [ann, bob, cy]

## QRY-03 · The extent is read when the first match is asked for

In [ ]:
pending = query.select("Contact", this.age.ge(40))
listed(store, loose)  # listed after the query was made, before it was read
assert list(pending) == [loose]

## QRY-04 · A query that cannot be right is refused when it is made

In [ ]:
with raises(AttributeError, match="no schema registered as 'Nope'"):
    query.select("Nope", this.has("name"))
with raises(TypeError, match="'Listed' is a relation"):
    query.select("Listed", this.has("name"))
with raises(ValueError, match="'this' is bound to each object; it is not a variable"):
    query.select("Contact", this.has("name"), {"this": ann})
with raises(ValueError, match="the rule cannot be a query: "):
    query.select("Contact", this.age.ge(E.variable("n")))
with raises(ValueError, match="'pow' is not a core operation"):
    query.select("Contact", E.operation("pow", this.age, 2).ge(1))
numbers = query.select("Contact", this.age)  # a rule must give a bool, found as objects are read
with raises(TypeError, match="a query's rule must give a bool"):
    list(numbers)

## QRY-05 · Scan is a store, and select asks a queryable store or scans another

In [ ]:
assert protocol_problems(query, Q.QueryableStore) == [] and protocol_problems(store, Q.QueryableStore) == ["select"]
assert (query.schema("Contact"), query.registered("Listed"), query.name_of(Contact)) == (Contact, store.registered("Listed"), "Contact")
assert query.names() == store.names() and query.extent("Contact") == store.extent("Contact")
assert query.singleton("book.Directory") is store.singleton("book.Directory")
assert query.member(ann, "name") == "ann" and query.builder("Contact").name("e").create().name == "e"
assert query.builder("Contact", ann).update() is ann and query.store is store
scanned = Q.select(store, "Contact", this.age.ge(18))  # a plain store is scanned; the update above reordered its entries
assert sorted(c.name for c in scanned) == ["ann", "dee"]


class Native:
    """A store that answers queries itself."""

    def select(self, name, rule, variables=None, unknown=False):
        return iter([("asked", name, variables, unknown)])


assert list(Q.select(Native(), "X", this.has("a"), {"v": 1}, True)) == [("asked", "X", {"v": 1}, True)]
Item = S.OfObject.Builder().ref().singleton("the.Item").properties(native("name")).create()
binding = Bd.Binding(Item, lambda t: Bd.State({"name": t.name}),
                     lambda s: type("Thing", (), {"name": s.values.get("name"), "accept": lambda self, v: Bd.accept(binding, self, v),
                                                  "identity": lambda self: id(self), "schema_name": lambda self: "Item",
                                                  "owner": lambda self: None})())
bound = Bd.OfStore({"Item": (Item, lambda instance=None: Bd.Builder(binding, instance))})
assert list(Q.select(bound, "Item", this.has("name").not_())) == [bound.singleton("the.Item")]  # bound classes too